In [34]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "/kaggle/input/datasets/abigailwiryokasa/testing/Drug.csv"

# Load the latest version
# df = kagglehub.load_dataset(
# KaggleDatasetAdapter.PANDAS,
# "thedevastator/drug-performance-evaluation",
# file_path,
#)

df = pd.read_csv(file_path)

print("First 5 records:", df.head())
print(df.info())


X = df.drop(columns=['EaseOfUse']).to_numpy()
y = df['EaseOfUse'].to_numpy()

# np.isnan(X).sum(axis=0)

/kaggle/input/datasets/abigailwiryokasa/testing/Drug.csv
First 5 records:                    Condition          Drug Indication Type      Reviews  \
0  Acute Bacterial Sinusitis  Levofloxacin   On Label   RX  994 Reviews   
1  Acute Bacterial Sinusitis  Levofloxacin   On Label   RX  994 Reviews   
2  Acute Bacterial Sinusitis  Moxifloxacin   On Label   RX  755 Reviews   
3  Acute Bacterial Sinusitis  Azithromycin   On Label   RX  584 Reviews   
4  Acute Bacterial Sinusitis  Azithromycin   On Label   RX  584 Reviews   

   Effective  EaseOfUse  Satisfaction  \
0       2.52       3.01          1.84   
1       2.52       3.01          1.84   
2       2.78       3.00          2.08   
3       3.21       4.01          2.57   
4       3.21       4.01          2.57   

                                         Information  
0  \r\n\t\t\t\t\tLevofloxacin is used to treat a ...  
1  \r\n\t\t\t\t\tLevofloxacin is used to treat a ...  
2  \r\n\t\t\t\t\t This is a generic drug. The ave...  
3  \r\n\

In [35]:
# By convention:

# X -> feature matrix, shape (n_samples, n_features)
# y -> label vector, shape (n_samples,)

#data uses pandas dataframe and not numpy
X = df.iloc[:, :-1] # every column except the last
y = df.iloc[:, -1] # only the last column

print(X.shape) # e.g. (500, 4) -> 500 samples, 4 features
print(y.shape) # e.g. (500,) -> 500 labels

(2219, 8)
(2219,)


In [36]:
#this only applies for numpy, would not work for pandas
# def train_test_split(X, y, test_ratio=0.2, seed=42):
#     rng = np.random.default_rng(seed)
#     n = X.shape[0]
#     indices = rng.permutation(n) # shuffle row indices
#     n_test = int(n * test_ratio)
#     test_idx = indices[:n_test]
#     train_idx = indices[n_test:]

#     return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

def train_test_split(X, y, test_ratio=0.2, seed=42):
    rng = np.random.default_rng(seed)

    n = len(X)
    indices = rng.permutation(n)  # shuffle row indices
    n_test = int(n * test_ratio)

    test_idx = indices[:n_test]
    train_idx = indices[n_test:]

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    return X_train, X_test, y_train, y_test


X_train, X_test, y_train, y_test = train_test_split(X, y)
X_train, X_test, y_train, y_test = train_test_split(X, y)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(1776, 8)
(443, 8)
(1776,)
(443,)


In [58]:
def standardize(X_train, X_test):
    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()

    numeric_cols = X_train.select_dtypes(include=["float64", "int64"]).columns

    mu = X_train[numeric_cols].mean()
    sigma = X_train[numeric_cols].std()

    X_train_scaled[numeric_cols] = (
        X_train[numeric_cols] - mu
    ) / sigma

    X_test_scaled[numeric_cols] = (
        X_test[numeric_cols] - mu
    ) / sigma

    return X_train_scaled, X_test_scaled

X_train_s, X_test_s = standardize(X_train, X_test)

print(X_train_s)
# print(X_train_s.head())
# print(X_train_s.mean())
# print(X_train_s.std())
print("------------------------------------------------------")
print(X_test_s)

                              Condition  \
1185                              fever   
563   Bacterial Urinary Tract Infection   
1801                       hypertension   
144                   Atopic Dermatitis   
1327    gastroesophageal reflux disease   
...                                 ...   
2189                            vertigo   
706                               edema   
1572                        hemorrhoids   
1633               hypercholesterolemia   
1672                       hypertension   

                                                   Drug Indication    Type  \
1185                     Doxylam-PE-DM-Acetaminophen-GG   On Label     OTC   
563                       Sulfamethoxazole-Trimethoprim   On Label      RX   
1801                                         Eplerenone   On Label      RX   
144                                        Fluocinolone   On Label      RX   
1327                                         Famotidine   On Label  RX/OTC   
...            

In [55]:
numeric_cols = X_train_s.select_dtypes(include=["float64", "int64"]).columns

print(X_train_s[numeric_cols].mean())
print(X_train_s[numeric_cols].std())

Effective       2.590520e-16
EaseOfUse       4.620928e-16
Satisfaction    1.035208e-16
dtype: float64
Effective       1.0
EaseOfUse       1.0
Satisfaction    1.0
dtype: float64
